# Basic Small Dataset

Generates data from a known generative model and compares the fitted
mean and component functions against the truth.

**Generative model:**
```
Y_i(t) = μ(t) + s_{i1} φ₁(t) + s_{i2} φ₂(t) + ε_i(t)
```
where μ(t) = sin(2πt), φ₁(t) = cos(2πt), φ₂(t) = sin(4πt),
s_{ij} ~ N(0,1), ε_i(t) ~ N(0, 0.05²)

In [ ]:
from __future__ import annotations

import math

import matplotlib.pyplot as plt
import torch

from irregpca import LiveLossPlotCallback, fit_irreg_pca

torch.manual_seed(0)

def true_mean(t: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * math.pi * t)

def true_phi1(t: torch.Tensor) -> torch.Tensor:
    return torch.cos(2 * math.pi * t)

def true_phi2(t: torch.Tensor) -> torch.Tensor:
    return torch.sin(4 * math.pi * t)

In [ ]:
# Simulate irregular observations
n_samples = 50
obs_per   = 15
sigma     = 0.05

ids_list, locs_list, vals_list = [], [], []
for i in range(n_samples):
    s1, s2 = torch.randn(2).tolist()
    t = torch.rand(obs_per, 1)
    y = (
        true_mean(t.squeeze())
        + s1 * true_phi1(t.squeeze())
        + s2 * true_phi2(t.squeeze())
        + sigma * torch.randn(obs_per)
    )
    ids_list.append(torch.full((obs_per,), float(i)))
    locs_list.append(t)
    vals_list.append(y)

sample_ids = torch.cat(ids_list)
locations  = torch.cat(locs_list, dim=0)
values     = torch.cat(vals_list)

In [ ]:
# Fit with live loss visualization
loss_cb = LiveLossPlotCallback(save_path="basic_small_dataset_loss.png")

result = fit_irreg_pca(
    sample_ids=sample_ids,
    locations=locations,
    values=values,
    n_components=2,
    epochs=600,
    lr=1e-3,
    patience=300,
    random_state=0,
    verbose=True,
    callbacks=[loss_cb],
)

In [ ]:
# Evaluate on a dense grid
grid = torch.linspace(0, 1, 300).unsqueeze(-1)
t    = grid.squeeze()

mu_hat   = result.mean(grid).cpu()
phi1_hat = result.component(0, grid).cpu()
phi2_hat = result.component(1, grid).cpu()

mu_true   = true_mean(t).cpu()
phi1_true = true_phi1(t).cpu()
phi2_true = true_phi2(t).cpu()

print(f"Component norms (should be ≈ sqrt(eigenvalue)):")
print(f"  φ₁: {result.component_norms()[0]:.4f}")
print(f"  φ₂: {result.component_norms()[1]:.4f}")
print(f"Gram matrix:\n{result.orthogonality_matrix().numpy().round(3)}")

In [ ]:
# Plot fitted vs true
x = t.numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

pairs = [
    (mu_hat.numpy(),   mu_true.numpy(),   "Mean  μ(t)"),
    (phi1_hat.numpy(), phi1_true.numpy(), "Component 1  φ₁(t)"),
    (phi2_hat.numpy(), phi2_true.numpy(), "Component 2  φ₂(t)"),
]

for ax, (fitted, truth, title) in zip(axes, pairs):
    ax.plot(x, truth,  lw=2, color="gray",     linestyle="--", label="true")
    ax.plot(x, fitted, lw=2, color="steelblue", label="fitted")
    ax.set_title(title)
    ax.set_xlabel("t")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    "Note: components are recovered up to sign and scale (‖φ̂‖ = √λ, not 1)",
    fontsize=9, color="gray",
)
plt.tight_layout()
plt.savefig("basic_small_dataset_example.png", dpi=150)
plt.show()